# From CM to trial results

## Introduction

Goal of this cookbook is to illustrate how one can post the material needed for a trial, run it and visualize the results.

Steps: 
1. Post a Computational Model
2. Create a Virtual Population (Vpop) Design
3. Generate a Vpop from the Vpop Design
4. Post a Protocol
5. Post a Data Table
6. Post a Trial
7. Run and monitor a Trial
8. Visualize the trial results

Linked resources: 
- [Folder on jinko](https://jinko.ai/project/e0fbb5bb-8929-439a-bad6-9e12d19d9ae4).

In [ ]:
# Jinko specifics imports & initialization
from jinko import JinkoClient

client = JinkoClient()
client.auth_check()

In [ ]:
# Cookbook specifics imports
import json
import os

import pandas as pd
import plotly.express as px

# Cookbook specifics constants:
# put here the constants that are specific to your cookbook like
# the reference to the Jinko items, the name of the model, etc.

# @param {"name":"folderId", "type": "string"}
# folder_id can be retrieved in the url, pattern is `https://jinko.ai/project/<project_id>?labels=<folder_id>`
folder_id = "5f573c8f-3f48-4c48-8257-f884837e5605"

folder = client.folders.get_by_id(folder_id)

resources_dir = os.path.normpath("./resources/run_a_trial")

model_file = os.path.join(resources_dir, "computational_model.json")
model_file_copy = os.path.join(resources_dir, "computational_model_copy.json")

solving_options_file = os.path.join(resources_dir, "solving_options.json")
vpop_file = os.path.join(resources_dir, "vpop.csv")
protocol_file = os.path.join(resources_dir, "protocol.json")
data_table_file = os.path.join(resources_dir, "data_table.csv")

# Step 1: Post a Computational Model

In [ ]:
# Load the model
with open(model_file, "r") as f:
    model_content = json.load(f)

# Load the solving options
with open(solving_options_file, "r") as f:
    solving_options_content = json.load(f)

payload = {
    "model": model_content,
    "solvingOptions": solving_options_content,
}

model = client.create_model_from_json(
    payload,
    name="simple tumor model",
    folder=folder_id,
)

# Get the URL of the resource
print(f"Resource link: {model.url}")

# Step 2: Create a Vpop Design

### Get model descriptors


In [ ]:
# get model baseline descriptors
baseline_descriptors = model.get_baseline_descriptors()
numeric_descriptors = baseline_descriptors.model_dump(by_alias=True)[
    "numericDescriptors"]

# build the default marginal distributions that will be used to create the vpop design
# we only select descriptors that are PatientDescriptorKnown, PatientDescriptorUnknown or PatientDescriptorPartiallyKnown
default_marginal_distributions = [
    {
        "distribution": {
            "highBound": descriptor["distribution"]["highBound"],
            "lowBound": descriptor["distribution"]["lowBound"],
            "tag": descriptor["distribution"]["tag"],
        },
        "id": descriptor["id"],
    }
    for descriptor in numeric_descriptors
    if any(
        tag in descriptor["inputTag"]
        for tag in [
            "PatientDescriptorKnown",
            "PatientDescriptorUnknown",
            "PatientDescriptorPartiallyKnown",
        ]
    )
]

# Creating a formatted message with the IDs from marginal distributions
ids_output = "IDs present in the Marginal Distributions:\n" + "\n".join(
    [distribution["id"] for distribution in default_marginal_distributions]
)

default_marginal_distributions

### Create a new list with the updated distributions

In [ ]:
# Define a dictionary for distribution settings
distribution_settings = {
    "initialTumorBurden": {"mean": 1.8, "stdev": 0.08, "base": 10, "tag": "LogNormal"},
    "kccCancerCell": {"mean": 12, "stdev": 0.5, "base": 10, "tag": "LogNormal"},
    "kGrowthCancerCell": {"mean": -3, "stdev": 0.05, "base": 10, "tag": "LogNormal"},
    "vmaxCancerCellDeath": {"mean": -1, "stdev": 0.05, "base": 10, "tag": "LogNormal"},
    "ec50Drug": {"mean": -3.5, "stdev": 0.05, "base": 10, "tag": "LogNormal"},
}

### Post the vpop design

In [ ]:
vpop_design = model.create_vpop_design_from_design(
    marginal_distributions=distribution_settings,
    name="vpop design for simple tumor model",
    folder=folder,
)

print(f"Resource link: {vpop_design.url}")

# Step 3: Generate a Vpop from the Vpop design

In [ ]:
vpop = vpop_design.generate_vpop(
    size=10,
    name="vpop for simple tumor model",
    folder=folder,
    seed=42,
    variance_reduction=True,
)

print(f"Resource link: {vpop.url}")

# Step 3 bis - not mandatory: Directly post a csv vpop

In [ ]:
vpop_from_csv = client.create_vpop_from_csv(
    csv_file_path=vpop_file,
    name="vpop for simple tumor model",
    folder=folder,
)

print(f"Resource link: {vpop_from_csv.url}")

# Step 4 : Post a Protocol

In [ ]:
# Load the protocol
with open(protocol_file, "r") as f:
    protocol_content = json.load(f)

protocol = model.create_protocol_design(
    protocol_content["scenarioArms"],
    name="protocol for simple tumor model",
    folder=folder,
)

arm_names = [arm["armName"] for arm in protocol_content["scenarioArms"]]

print(f"Resource link: {protocol.url}")

# Step 5: Post a Data Table

In [ ]:
data_table = client.create_data_table_from_csv(
    csv_file_path=data_table_file,
    name="data table for simple tumor model",
    folder=folder,
)

print(f"Resource link: {data_table.url}")

# Step 6: Post a Trial

In [ ]:
trial = model.create_trial(
    vpop=vpop_from_csv,
    protocol=protocol,
    data_tables=[data_table],
    name="trial for simple tumor model",
    folder=folder,
)

print(f"Resource link: {trial.url}")

# Step 7 : Run and monitor a trial


### Run the trial

In [ ]:
trial.run()

### get trial status


In [ ]:
trial.wait_until_completed()

# Step 8 : Visualize the trial results 

In [ ]:
responseSummary = trial.output_ids()

print("Available time series:\n", responseSummary, "\n")

In [ ]:
# replace here by the time series ids list you want
idsForTimeSeries = ["tumorBurden"]

print("Retrieving time series data...")
dfTimeSeries = trial.results.timeseries(
    {ts: arm_names for ts in idsForTimeSeries}
).to_dataframe()

In [ ]:
display(dfTimeSeries.head(5))

# Extract unique patient IDs
unique_patient_ids = dfTimeSeries["Patient Id"].unique().tolist()

# Display unique patient IDs
print(unique_patient_ids)

In [ ]:
# Filter data for the first patient
patient_data = dfTimeSeries[dfTimeSeries["Patient Id"]
                            == unique_patient_ids[0]]

# Plot using Plotly
fig = px.line(
    patient_data,
    x="Time",
    y="Value",
    color="Arm",
    title="Time Series of Tumor Burden",
    labels={"Time": "Time (seconds)", "Value": "Tumor Burden Value"},
    markers=True,
)

fig.show()